# Week 2 - ML Task Framing

**Name:** Muhammad Usman Shakir

**Internship:** FlyRank AI

**Lane:** Content Refresh

This notebook explains how the Content Refresh problem can be framed as a Machine Learning task.


## 1. My Lane as an ML Task

### Task Type
The Content Refresh problem is best framed as a **scoring (regression/ranking)** task.

The objective is to estimate which pages are most likely to benefit from being refreshed. Instead of manually reviewing thousands of pages, a machine learning model can assign each page a score representing its refresh priority.

Pages with higher scores can be prioritized by content teams for updates.

## 2. Target or Proxy

The dataset does not contain a direct "needs refresh" label.

Instead, a proxy target can be created using existing metrics such as:

- Click Through Rate (CTR)
- Average Position
- Impressions
- Clicks

Pages with low CTR and poor search position may benefit from content updates, making these metrics useful indicators for prediction.

## 3. Success Metric

The success of the machine learning model can be measured using:

- Mean Absolute Error (MAE) or RMSE if predicting a numerical score.
- Precision@K or NDCG if ranking pages by refresh priority.

From a business perspective, success means identifying pages that gain higher CTR, better search rankings, and increased organic traffic after content updates.


In [5]:
!git clone https://github.com/muhammadusmanshakir/flyrank-ml-internship.git

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 75, done.
remote: Counting objects: 100% (75/75), done.
remote: Compressing objects: 100% (69/69), done.
remote: Total 75 (delta 26), reused 29 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (75/75), 1.64 MiB | 9.98 MiB/s, done.
Resolving deltas: 100% (26/26), done.


In [6]:
%cd flyrank-ml-internship

/content/flyrank-ml-internship/flyrank-ml-internship


In [7]:
!ls data/raw

content_refresh_anonymized.csv


In [8]:
import pandas as pd

# Load the dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Show the first 5 rows
df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [9]:
print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


## 4. The Unit of Analysis

The unit of analysis is a **single content page**.

Each row in the dataset represents one webpage and contains information about its search performance, content characteristics, user engagement, and traffic trends.

These features help determine whether a page is a good candidate for content refresh.

In [10]:
# Display the first five rows
df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [11]:
# Display important columns for the Content Refresh lane

df[[
    "content_id",
    "content_type",
    "word_count",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "trend_direction",
    "trend_pct"
]].head(10)

,content_id,content_type,word_count,impressions_90d,clicks_90d,ctr,avg_position,trend_direction,trend_pct
0,content_304f48230142,keyword article,3221.0,3803,29,0.76,10.6,down,-41.4
1,content_a1fb4e703a9e,keyword article,2481.0,15320,7,0.05,20.3,down,-57.7
2,content_9aa793d4d895,keyword article,3515.0,12581,11,0.09,36.5,down,-60.9
3,content_331d6c4de07b,keyword article,NaN,11751,58,0.49,6.2,stable,-13.8
4,content_d99b7a2d90ca,keyword article,2803.0,19140,24,0.13,44.0,down,-34.7
5,content_d4084a4bc775,keyword article,3080.0,3970,1,0.03,8.5,down,-38.9
6,content_9a34b442b552,keyword article,3059.0,20,0,0.00,7.0,down,-92.3
7,content_a63219c6e95a,keyword article,NaN,1724,1,0.06,21.2,stable,0.6
8,content_5e6c160719bc,keyword article,3807.0,32574,29,0.09,46.0,down,-58.8
9,content_c27558df2b0c,keyword article,NaN,1240,2,0.16,4.9,down,-29.2


## 5. Why ML Beats a Fixed Rule

A fixed rule might recommend refreshing every page with low CTR or a poor average search position.

However, page performance depends on many factors working together, including search volume, impressions, clicks, content age, freshness, engagement rate, traffic trends, and search position.

Machine Learning can learn patterns from all these features simultaneously and produce better recommendations than a simple threshold-based rule. This helps prioritize pages that are most likely to benefit from a content refresh.

## 6. Self Check

✅ Identified the ML task type.

✅ Defined a target/proxy for prediction.

✅ Selected appropriate success metrics.

✅ Displayed the unit of analysis using the real dataset.

✅ Explained why Machine Learning is more suitable than fixed rules.

✅ Connected the model output to a real business action (prioritizing content for refresh).

In [12]:
print("Total pages:", len(df))

print("Content Types:")
print(df["content_type"].value_counts())

print("\nTrend Direction:")
print(df["trend_direction"].value_counts())

print("\nAverage CTR:", round(df["ctr"].mean(), 3))
print("Average Position:", round(df["avg_position"].mean(), 2))

Total pages: 30000
Content Types:
content_type
keyword article       27207
feedly article         2096
comparison article      697
Name: count, dtype: int64

Trend Direction:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Average CTR: 0.511
Average Position: 16.34
